# Historic-area CLIP: an offline-first spatial reasoning path

This lesson asks whether street-level scenes are more similar to text prompts describing historic, modern, mixed, or open-space settings. The default path uses a deterministic sample from the 582 tracked Barcelona images and a cached CLIP result table, so it needs no Google key, network, or model download. Set `RUN_GOOGLE_API = True` only when you intentionally want new Street View images; set `RUN_CLIP_MODEL = True` only when the CLIP model is already cached or you accept a model download.

![Decision tree and spatial chain](../../../docs/figs/clip_historic_decision_spatial.svg)

A CLIP score is image–text similarity. It is a model-derived candidate signal, not a verified historic-status label. Coordinates, headings, image filenames, and scores stay joined through one manifest before the mapped index is calculated.

## Step 1: Choose a historic area and sample points along the streets

In [ ]:
import ast
import numpy as np
import pandas as pd
import re
import os
import folium
import osmnx as ox
from shapely.geometry import box
from tqdm import tqdm
from pathlib import Path

from ccai9012.paths import STARTER_KITS_DIR
RUN_GOOGLE_API = False
from ccai9012.svi_utils import (
    build_svi_image_manifest,
    deterministic_image_sample,
    join_clip_results,
)

In [ ]:
from ccai9012 import svi_utils, viz_utils
from ccai9012.multi_modal_utils import CLIPClassifier

In [ ]:
# Define the bounding box coordinates (Barcelona – Eixample + 22@ Innovation District, Spain)
bbox = (2.1780, 41.3975, 2.2030, 41.4065) # (west, south, east, north)

# Point sampling through OSMnx is also network-backed, so keep it in the opt-in branch.
if RUN_GOOGLE_API:
    m, points = viz_utils.sample_street_points_map(bbox)
    display(m)
else:
    m, points = None, []
    print("Offline sample mode: no route/point API call will be made.")

## Step 2: Choose local sample or Google Street View API

### Default branch: deterministic local sample

The manifest is the contract between a point, its heading, its image, and later CLIP results. Sorting before sampling makes the selection reproducible even if the filesystem returns files in a different order. The API branch below is explicit opt-in and is never constructed in sample mode.

In [ ]:
CLIP_DIR = STARTER_KITS_DIR / "3_multimodal_reasoning" / "clip_historic"
IMAGE_DIR = CLIP_DIR / "images"
RUN_GOOGLE_API = False
RUN_CLIP_MODEL = False  # True only with a cached/approved CLIP model
SAMPLE_SIZE = 24
SAMPLE_SEED = 42

manifest = build_svi_image_manifest(IMAGE_DIR)
sample_manifest = deterministic_image_sample(manifest, SAMPLE_SIZE, SAMPLE_SEED)
print(f"Manifest: {len(manifest)} tracked images; sample: {len(sample_manifest)} images")
sample_manifest.head()

In [ ]:
if RUN_GOOGLE_API:
    downloader = svi_utils.GoogleSVIDownloader(save_dir=str(IMAGE_DIR))
else:
    downloader = None
    print("Sample mode: no Google key requested and no network call will be made.")

In [ ]:
if RUN_GOOGLE_API:
    # Camera headings: 0 = North, 90 = East, etc.
    HEADINGS = [90, 270]
    for lat, lon in tqdm(points, desc="Downloading SVI"):
        for heading in HEADINGS:
            filepath = downloader.download_svi(lat, lon, heading=heading)
            if not filepath:
                print(f"Failed: lat={lat:.6f}, lon={lon:.6f}, heading={heading}")
    manifest = build_svi_image_manifest(IMAGE_DIR)
    sample_manifest = deterministic_image_sample(manifest, SAMPLE_SIZE, SAMPLE_SEED)
else:
    print("API mode is disabled. The local manifest remains active.")

## Step 3: Classfication with CLIP

In [ ]:
# Use a cached result table by default; model inference is an explicit opt-in.
CACHED_RESULTS = CLIP_DIR / "inference_results.csv"
if RUN_CLIP_MODEL:
    clip = CLIPClassifier(image_dir=str(IMAGE_DIR))
else:
    clip = None
    print("Offline sample mode: using the tracked CLIP result cache.")

In [ ]:
# Define visual prompt categories (tailored to Barcelona region)
text_prompts = [
    "historic or traditional European buildings made of brick or stone",  # 0 - Historic
    "modern buildings with glass, metal, concrete or painted facade",                            # 1 - Modern
    "a combination of historic and modern buildings in the same scene, showing contrast in architectural styles",             # 2 - Mixed
    "an open public space such as a plaza, park, or square, with no visible buildings in the frame, featuring trees or sky"    # 3 - Open Space
]

if RUN_CLIP_MODEL:
    # Classify only the selected files; this bounds the opt-in model run.
    rows = [clip.classify_image(row["image_path"], text_prompts) for _, row in sample_manifest.iterrows()]
    df_results = pd.DataFrame([row for row in rows if row is not None])
    df_results = join_clip_results(df_results, sample_manifest)
else:
    cached = pd.read_csv(CACHED_RESULTS)
    cached = cached[cached["filename"].isin(sample_manifest["filename"])].copy()
    df_results = join_clip_results(cached, sample_manifest)

# Cached CSV stores the score list as text; parse it once before scoring.
df_results["all_scores"] = df_results["all_scores"].apply(ast.literal_eval)
df_results[["filename", "latitude", "longitude", "heading", "label_id"]].head()

In [ ]:
# Show a reproducible sample. The image path comes from the manifest, not parsing.
from PIL import Image
for index in range(min(4, len(df_results))):
    row = df_results.iloc[index]
    display(Image.open(row["image_path"]))
    print(f"{row['filename']}: {row['label_text']} (score={row['confidence']:.2f})")

## Step 4: Generate Historic Index Heatmap

In [ ]:
import ast
from folium.plugins import HeatMap

In [ ]:
# Coordinates and heading were attached by join_clip_results above.
assert df_results[["latitude", "longitude", "heading"]].notna().all().all()
df_results[["filename", "latitude", "longitude", "heading"]].head()

In [ ]:
# Define weights for label_id: 0->3, 1->2, 2->1, 3->0 (ignore)
weights = {0: 3, 1: 2, 2: 1, 3: 0}

def weighted_score(scores):
    # scores is a list of confidences corresponding to [id0, id1, id2, id3]
    total_weight = 3 + 2 + 1  # sum of weights for id 0,1,2
    score_sum = scores[0]*3 + scores[1]*2 + scores[2]*1
    return score_sum / total_weight

# Calculate weighted score per row
df_results['weighted_score'] = df_results['all_scores'].apply(weighted_score)

# Group by lat, lon and average weighted scores over headings
grouped = df_results.groupby(['latitude', 'longitude'])['weighted_score'].mean().reset_index()
grouped = grouped.rename(columns={
    "latitude":"latitude",
    "longitude":"longitude"
})
grouped

In [ ]:
m = viz_utils.plot_points(grouped, value_col='weighted_score')
display(m)